In [0]:
catalog = "firstdeproject"
schema = "silver"

tables = [t.tableName for t in spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect() if not t.isTemporary]

for t in tables:
    print(f"\n=== {catalog}.{schema}.{t} ===")
    spark.table(f"{catalog}.{schema}.{t}").show(5, truncate=False)

In [0]:

from pyspark.sql import functions as F, Window

CATALOG, SILVER, GOLD = "firstdeproject", "silver", "gold"
META = ["_source_folder", "_source_file", "_ingested_at"]

def silver(t):
    return spark.table(f"{CATALOG}.{SILVER}.{t}")

def save_gold(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable(f"{CATALOG}.{GOLD}.{name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD}")


In [0]:
# Key uniqueness: each should return 0 rows if the key is unique
for t, k in [("cust_info", "cst_id"), ("cust_az12", "cid_clean"), ("loc_a101", "cid_clean")]:
    dups = silver(t).groupBy(k).count().filter("count > 1").count()
    print(f"{t}.{k}: {dups} duplicated keys")

# prd_info is SCD2 (history rows), so several rows per prd_key is expected
silver("prd_info").groupBy("prd_key").count().orderBy(F.desc("count")).show(5)


# Do product keys line up between prd_info and sales_details?
# If line up, distinct prd_key from prd_info, sales_details and their inner join should have same count
p = silver("prd_info").select("prd_key").distinct()
s = silver("sales_details").select("sls_prd_key").distinct()
print("sales product keys found in prd_info:",
      s.join(p, s.sls_prd_key == p.prd_key, "inner").count(), "of", s.count())



In [0]:

# 1. dim_customers
# CRM (cust_info) is the master; ERP tables only enrich (birthdate, country, gender fallback).

# Normalize gender to consistent format
def norm_gender(c):
    c = F.upper(F.trim(c))
    return (F.when(c.isin("M", "MALE"), "Male")
             .when(c.isin("F", "FEMALE"), "Female"))



# ensure cst_id is unique by retaining the latest unique one 
# ensure joining key, cst_id has consistent format (string)
ci = (silver("cust_info")
      .filter(F.col("cst_id").isNotNull()) # Filter out null cst_id
      .withColumn("rn", F.row_number().over(
          Window.partitionBy("cst_id").orderBy(F.col("cst_create_date").desc())))
      .filter("rn = 1").drop("rn", *META)
      .withColumn("cst_id", F.col("cst_id").cast("string")))

# alias renames the joining keys to have same name in both tables
az = (silver("cust_az12")
      .select(F.col("cid_clean").cast("string").alias("cst_id"),
              F.col("BDATE").alias("birthdate"),
              F.col("cst_gndr").alias("erp_gender"))
      .dropDuplicates(["cst_id"]))

loc = (silver("loc_a101")
       .select(F.col("cid_clean").cast("string").alias("cst_id"),
               F.col("CNTRY").alias("country"))
       .dropDuplicates(["cst_id"]))

dim_customers = (
    ci.join(az, "cst_id", "left")
      .join(loc, "cst_id", "left")
      .select(
          F.row_number().over(Window.orderBy("cst_id")).alias("customer_key"),   # surrogate key
          F.col("cst_id").alias("customer_id"),
          F.col("cst_key").alias("customer_number"),
          F.col("cst_firstname").alias("first_name"),
          F.col("cst_lastname").alias("last_name"),
          F.col("cst_fullname").alias("full_name"),
          F.when(F.upper("cst_marital_status") == "M", "Married")
           .when(F.upper("cst_marital_status") == "S", "Single")
           .otherwise("n/a").alias("marital_status"),
          F.coalesce(norm_gender(F.col("cst_gndr")),        # CRM first, ERP as fallback
                     norm_gender(F.col("erp_gender")),
                     F.lit("n/a")).alias("gender"),
          F.col("birthdate"),
          F.coalesce("country", F.lit("n/a")).alias("country"),
          F.col("cst_create_date").alias("create_date"),
      ))

save_gold(dim_customers, "dim_customers")



In [0]:
print(dim_customers.show(5))